In [ ]:
# Parse Kafka JSON messages into a structured Spark DataFrame and display the results in the console.

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import (
    ArrayType,
    DoubleType,
    LongType,
    StringType,
    StructField,
    StructType,
)


KAFKA_BOOTSTRAP_SERVERS = "YOUR_KAFKA_PRIVATE_IP:9092"
KAFKA_TOPIC = "stock-trades-raw"

CHECKPOINT_LOCATION = (
    "/home/ubuntu/stock-market-streaming/"
    "checkpoints/parsed_console_test"
)


TRADE_SCHEMA = StructType(
    [
        StructField("schema_version", StringType(), True),
        StructField("event_id", StringType(), True),
        StructField("event_type", StringType(), True),
        StructField("source", StringType(), True),
        StructField("symbol", StringType(), True),
        StructField("price", DoubleType(), True),
        StructField("volume", DoubleType(), True),
        StructField(
            "trade_conditions",
            ArrayType(StringType()),
            True,
        ),
        StructField("event_timestamp_ms", LongType(), True),
        StructField("event_timestamp_utc", StringType(), True),
        StructField("ingestion_timestamp_utc", StringType(), True),
    ]
)


def main() -> None:
    spark = (
        SparkSession.builder
        .appName("StockMarketParsedKafkaConsumer")
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("WARN")

    raw_stream = (
        spark.readStream
        .format("kafka")
        .option(
            "kafka.bootstrap.servers",
            KAFKA_BOOTSTRAP_SERVERS,
        )
        .option("subscribe", KAFKA_TOPIC)
        .option("startingOffsets", "earliest")
        .load()
    )

    string_stream = raw_stream.selectExpr(
        "CAST(key AS STRING) AS message_key",
        "CAST(value AS STRING) AS message_value",
        "topic",
        "partition",
        "offset",
        "timestamp AS kafka_timestamp",
    )

    parsed_stream = (
        string_stream
        .withColumn(
            "trade",
            from_json(
                col("message_value"),
                TRADE_SCHEMA,
            ),
        )
        .select(
            col("trade.schema_version"),
            col("trade.event_id"),
            col("trade.event_type"),
            col("trade.source"),
            col("trade.symbol"),
            col("trade.price"),
            col("trade.volume"),
            col("trade.trade_conditions"),
            col("trade.event_timestamp_ms"),
            col("trade.event_timestamp_utc"),
            col("trade.ingestion_timestamp_utc"),
            col("message_key"),
            col("topic"),
            col("partition"),
            col("offset"),
            col("kafka_timestamp"),
        )
    )

    query = (
        parsed_stream.writeStream
        .format("console")
        .outputMode("append")
        .option("truncate", "false")
        .option(
            "checkpointLocation",
            CHECKPOINT_LOCATION,
        )
        .trigger(processingTime="5 seconds")
        .start()
    )

    print("Parsed Spark consumer is running.")
    print("Press Ctrl+C to stop.")

    query.awaitTermination()


if __name__ == "__main__":
    main()